In [ ]:
%pip install -q -r requirements.txt
!python -m py_compile realesrgan.py realesrgan_fast.py realesrgan_fast_entry.py realesrgan_line_entry.py enhance/*.py
!python realesrgan_line_entry.py --help >/dev/null


In [ ]:
INPUT_VIDEO = "cm_4.mp4"
OUTPUT_VIDEO = "realesrgan_basicvsrpp.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

# BasicVSR++ compressed-video enhancement before Real-ESRGAN.
BASICVSRPP = True
BASICVSRPP_TRACK = 1
BASICVSRPP_MODEL_PATH = ""
BASICVSRPP_GPU = 0
BASICVSRPP_FP16 = True
BASICVSRPP_CLIP_LENGTH = 9
BASICVSRPP_CLIP_OVERLAP = 2
BASICVSRPP_TILE_SIZE = 512
BASICVSRPP_TILE_PAD = 32
BASICVSRPP_STRENGTH = 1.0
BASICVSRPP_SCENE_THRESHOLD = 0.30

# Real-ESRGAN single-4090 shared-memory tuning.
AUTO_TILE = True
MAX_TILE_SIZE = 1536
AUTO_BATCH = True
MAX_BATCH_SIZE = 32
TILE_SIZE = 256
TILE_PAD = 10
TILE_VERIFY_COVERAGE = False
BATCH_SIZE = 4
GPU_IDS = "0"

# v5.0: enhance only coherent low-contrast lines and protect strong edges.
LOW_CONTRAST_LINES = True
LINE_STRENGTH = 1.0
LINE_GRADIENT_MIN = 0.004
LINE_GRADIENT_MAX = 0.025
LINE_PROTECT_GRADIENT = 0.040
LINE_COHERENCE = 0.45
LINE_GUIDED_RADIUS = 4
LINE_GUIDED_EPS = 4.0e-4
LINE_TEMPORAL = 0.65
LINE_MAX_DELTA = 0.025

COLOR_POLICY = "preserve"
HDR_POLICY = "reject"
VIDEO_CODEC = "libx265"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
import os
import shlex
import subprocess
import sys

command = [
    sys.executable, "realesrgan_line_entry.py",
    "--input", INPUT_VIDEO, "--output", OUTPUT_VIDEO,
    "--model", MODEL, "--model-path", MODEL_PATH,
    "--scale", str(SCALE), "--fps", FPS,
    "--fp16", "--channels-last",
    "--basicvsrpp" if BASICVSRPP else "--no-basicvsrpp",
    "--basicvsrpp-track", str(BASICVSRPP_TRACK),
    "--basicvsrpp-model-path", BASICVSRPP_MODEL_PATH,
    "--basicvsrpp-gpu", str(BASICVSRPP_GPU),
    "--basicvsrpp-fp16" if BASICVSRPP_FP16 else "--no-basicvsrpp-fp16",
    "--basicvsrpp-clip-length", str(BASICVSRPP_CLIP_LENGTH),
    "--basicvsrpp-clip-overlap", str(BASICVSRPP_CLIP_OVERLAP),
    "--basicvsrpp-tile-size", str(BASICVSRPP_TILE_SIZE),
    "--basicvsrpp-tile-pad", str(BASICVSRPP_TILE_PAD),
    "--basicvsrpp-strength", str(BASICVSRPP_STRENGTH),
    "--basicvsrpp-scene-threshold", str(BASICVSRPP_SCENE_THRESHOLD),
    "--auto-tile" if AUTO_TILE else "--no-auto-tile",
    "--max-tile-size", str(MAX_TILE_SIZE),
    "--auto-batch" if AUTO_BATCH else "--no-auto-batch",
    "--max-batch-size", str(MAX_BATCH_SIZE),
    "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD),
    "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
    "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
    "--low-contrast-lines" if LOW_CONTRAST_LINES else "--no-low-contrast-lines",
    "--line-strength", str(LINE_STRENGTH),
    "--line-gradient-min", str(LINE_GRADIENT_MIN),
    "--line-gradient-max", str(LINE_GRADIENT_MAX),
    "--line-protect-gradient", str(LINE_PROTECT_GRADIENT),
    "--line-coherence", str(LINE_COHERENCE),
    "--line-guided-radius", str(LINE_GUIDED_RADIUS),
    "--line-guided-eps", str(LINE_GUIDED_EPS),
    "--line-temporal", str(LINE_TEMPORAL),
    "--line-max-delta", str(LINE_MAX_DELTA),
    "--color-policy", COLOR_POLICY, "--hdr-policy", HDR_POLICY,
    "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg", "--ffprobe-bin", "ffprobe",
]
print("[command]", shlex.join(command), flush=True)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(command, check=True, env=env)
